# Лекция 10. HTTP-запросы и работа с API в Python


*Лекция прочитана Иваном Кисляковым. Разрешено использование в некоммерческих образовательных целях.*


## 1. Как программы обмениваются данными

### Клиент и сервер

- **Клиент** — программа, которая обращается за ресурсом или действием. Например, браузер или Python-скрипт.
- **Сервер** — программа, которая принимает запрос и отправляет ответ.
- **HTTP** (Hypertext Transfer Protocol) — протокол обмена запросами и ответами. Веб-сервисы обычно работают по HTTPS: HTTP поверх защищённого соединения.

```text
       КЛИЕНТ (Python)                       СЕРВЕР (API)
               |                                  |
               | ---- GET /todos/1 HTTP ------->  |
               |                                  |
               | <--- 200 OK + JSON ------------  |
               |                                  |
```

**URL** указывает, куда обращаемся. Пример: `https://jsonplaceholder.typicode.com/todos/1`.

Разберём его:
- `https` — схема обращения;
- `jsonplaceholder.typicode.com` — имя хоста;
- `/todos/1` — путь к ресурсу.

### Методы HTTP

| Метод | Обычное назначение | Пример |
|---|---|---|
| `GET` | Получить ресурс | Получить список задач |
| `POST` | Отправить данные на обработку или создать ресурс | Добавить задачу |
| `PUT` / `PATCH` | Обновить ресурс | Изменить название задачи |
| `DELETE` | Удалить ресурс | Удалить задачу |

Назначение метода — договорённость интерфейса API. Например, GET не должен изменять состояние сервера.

### Из чего состоит HTTP-ответ?

1. **Статус** (например, `200 OK` или `404 Not Found`).
2. **Заголовки** (например, `Content-Type`).
3. **Тело ответа**: текст, JSON, изображение, файл и т. п.

**Вопрос для обсуждения.** Если клиент получил страницу с текстом «Не найдено», означает ли это, что HTTP-запрос обязательно завершился с кодом 404? Почему?

---
## 2. Первый запрос с `requests`

`requests` — сторонняя библиотека Python, которая упрощает работу с HTTP.

```python
%pip install requests
```

In [2]:
import requests

url = "https://httpbin.org/get"
response = requests.get(url, timeout=10)

print("Тип объекта:", type(response))
print("Статус:", response.status_code)
print("Адрес:", response.url)

Тип объекта: <class 'requests.models.Response'>
Статус: 200
Адрес: https://httpbin.org/get


В переменной `response` находится **объект ответа**, а не просто текст. У него есть разные свойства и методы.

| Выражение | Что получаем |
|---|---|
| `response.status_code` | Числовой HTTP-статус |
| `response.headers` | Заголовки ответа (объект, похожий на словарь) |
| `response.text` | Тело ответа как строку |
| `response.content` | Тело ответа как байты |
| `response.url` | Итоговый URL запроса |
| `response.json()` | Декодированные JSON-данные (если тело содержит корректный JSON) |

> Метод `response.json()` **не проверяет успешность HTTP-статуса** и может вызвать ошибку, если тело ответа не является корректным JSON.

In [ ]:
print("Content-Type:", response.headers.get("Content-Type"))
print("Длина ответа (байт):", len(response.content))
print("Первые 250 символов:")
print(response.text[:250])

### Исследуем объект самостоятельно

`dir()` показывает доступные имена атрибутов и методов, `help()` — документацию. Это полезно, когда вы впервые встретили незнакомую библиотеку.

In [ ]:
print("Есть ли метод json?", "json" in dir(response))
print("Есть ли атрибут headers?", "headers" in dir(response))

# Для подробного описания раскомментируйте одну строку:
# help(requests.get)
# help(response.json)

**Мини-задание 1 (2 минуты).** Измените адрес на `https://httpbin.org/status/404`. Выведите статус и длину тела. Почему сам вызов `requests.get()` может отработать без исключения, хотя код ответа — 404?

**Вывод:** наличие объекта `Response` ещё не означает, что сервер успешно обработал запрос.

---
## 3. Параметры GET-запроса

Параметры часто передаются в **строке запроса** (`query string`) — части URL после `?`.

```text
https://example.org/search?city=Moscow&year=2026
                           ^^^^^^^^^^^^^^^^^^^
                           параметры запроса
```

Вместо ручного составления URL передайте словарь в `params`. Библиотека закодирует значения и сформирует адрес.

In [3]:
url = "https://httpbin.org/get"
params = {
    "city": "Moscow",
    "year": 2026,
    "topic": "Python requests",
}

response = requests.get(url, params=params, timeout=10)
response.raise_for_status()

print("Итоговый URL:", response.url)
print("Параметры, полученные сервером:", response.json()["args"])

Итоговый URL: https://httpbin.org/get?city=Moscow&year=2026&topic=Python+requests
Параметры, полученные сервером: {'city': 'Moscow', 'topic': 'Python requests', 'year': '2026'}


В ответе учебного сервиса `httpbin` поле `args` содержит разобранные параметры. Обратите внимание: значения из URL обычно читаются как **строки**, даже если в Python мы передавали число.

**Мини-задание 2 (2 минуты).** Передайте параметры `name`, `age`, `course`, затем проверьте, что сервер получил все три значения. Как сервер видит `age` — как целое число или строку?

### Cтатическая карта

Для запроса к API статических карт Яндекса можно передать координаты, масштаб и ключ доступа `https://static-maps.yandex.ru/v1`.


In [ ]:
from IPython.display import Image, display

api_key = "YANDEX_MAPS_API_KEY"


maps_url = "https://static-maps.yandex.ru/v1"
maps_params = {
    "ll": "37.648297,55.753636",  # долгота, широта
    "z": 14,
    "lang": "ru_RU",
    "apikey": api_key,
}

maps_response = requests.get(maps_url, params=maps_params, timeout=15)
maps_response.raise_for_status()

print("Тип ответа:", maps_response.headers.get("Content-Type"))
# При успешном получении изображения отобразим его:
if "image/" in maps_response.headers.get("Content-Type", ""):
    display(Image(data=maps_response.content))
else:
    print("Сервер вернул не изображение; проверьте параметры и условия API.")

---
## 4. API, JSON и отправка данных

### Что такое API?

**API** (Application Programming Interface) — определённый интерфейс, по которому одна программа взаимодействует с другой. Веб-API обычно описывает:
- адреса ресурсов (эндпоинты);
- допустимые методы и параметры;
- структуру запроса и ответа;
- ошибки и правила авторизации.

Например, вместо самостоятельного хранения данных о задачах программа может получать их от сервиса.

**Преимущества для клиента:** использование готовых функций и актуальных данных; не требуется реализовывать всё с нуля.  
**Преимущества для сервиса:** единый интерфейс интеграции, контроль доступа и возможность предоставлять функциональность другим приложениям.

### JSON ↔ Python

JSON — текстовый формат обмена структурированными данными. Он похож на словари и списки Python:

```json
{
  "userId": 1,
  "id": 1,
  "title": "Пример задачи",
  "completed": false
}
```

Значение `false` в JSON превращается в `False` в Python, JSON-объект — в словарь, массив — в список.

In [4]:
url = "https://jsonplaceholder.typicode.com/todos/1"
response = requests.get(url, timeout=10)
response.raise_for_status()

todo = response.json()
print(type(todo))
print("Название:", todo["title"])
print("Выполнена:", todo["completed"])
print("Пользователь:", todo["userId"])

<class 'dict'>
Название: delectus aut autem
Выполнена: False
Пользователь: 1


### Получаем список объектов

Теперь сервер возвращает не один словарь, а список словарей. Мы можем использовать привычные циклы, условия и функции.

In [5]:
response = requests.get(
    "https://jsonplaceholder.typicode.com/todos",
    params={"userId": 1},
    timeout=10,
)
response.raise_for_status()

todos = response.json()
completed = [todo for todo in todos if todo["completed"]]

print("Всего задач:", len(todos))
print("Выполнено:", len(completed))
print("Не выполнено:", len(todos) - len(completed))

for todo in todos[:3]:
    print(todo["id"], "—", todo["title"])

Всего задач: 20
Выполнено: 11
Не выполнено: 9
1 — delectus aut autem
2 — quis ut nam facilis et officia qui
3 — fugiat veniam minus


**Мини-задание 3 (3 минуты).** Поменяйте `userId` на 2 и выведите процент завершённых задач с точностью до одного знака после запятой. Если список пуст, не делите на ноль.

### Чем POST отличается от GET?

GET обычно получает информацию, а POST — отправляет данные в теле запроса. Аргумент `json=` самостоятельно сериализует словарь в JSON и задаёт подходящий заголовок `Content-Type`.

`JSONPlaceholder` — учебный сервис: он имитирует создание записи и возвращает результат, но **не сохраняет новую задачу постоянно**.

In [6]:
new_todo = {
    "userId": 1,
    "title": "Разобраться с HTTP и API",
    "completed": False,
}

response = requests.post(
    "https://jsonplaceholder.typicode.com/todos",
    json=new_todo,
    timeout=10,
)
response.raise_for_status()

print("Статус:", response.status_code)  # обычно 201 Created
print("Ответ сервера:", response.json())

Статус: 201
Ответ сервера: {'userId': 1, 'title': 'Разобраться с HTTP и API', 'completed': False, 'id': 201}


**Вопросы на понимание:**

1. Почему `response.json()` часто удобнее, чем `response.text`?
2. Почему `json=new_todo` не то же самое, что `params=new_todo`?
3. Почему возвращённый идентификатор задачи не доказывает, что данные были сохранены надолго?

---
## 5. Ошибки и надёжность сетевого кода

### Частые HTTP-статусы

| Код | Значение | Как трактовать |
|---:|---|---|
| 200 | OK | Запрос успешно обработан |
| 201 | Created | Ресурс создан (или имитация создания в учебном API) |
| 400 | Bad Request | Ошибка в запросе клиента |
| 401 | Unauthorized | Требуется аутентификация или она не пройдена |
| 403 | Forbidden | Доступ запрещён |
| 404 | Not Found | Ресурс не найден |
| 429 | Too Many Requests | Превышен лимит запросов |
| 500 | Internal Server Error | Ошибка на стороне сервера |

- `response.status_code` позволяет узнать код.
- `response.raise_for_status()` выбрасывает `requests.exceptions.HTTPError` при ответах 4xx или 5xx.
- `timeout=10` ограничивает ожидание сетевых операций; это не гарантия общего времени выполнения ровно в 10 секунд.
- `try/except` позволяет отдельно обработать проблемы с сетью или HTTP-статусом.

> В реальной программе не стоит делать десятки запросов без ограничения частоты или повторять запрос бесконечно.

In [ ]:
from requests.exceptions import HTTPError, RequestException, Timeout

url = "https://httpbin.org/status/404"

try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    print(response.text[:100])
except HTTPError as error:
    print("HTTP-ошибка:", error.response.status_code)
except Timeout:
    print("Сервер не ответил вовремя")
except RequestException as error:
    print("Сетевая ошибка:", error)

### Практическое правило

Для учебных API обычно достаточно:

1. Задать `timeout`.
2. Вызвать `raise_for_status()` до обработки успешного ответа.
3. При необходимости обернуть сетевой вызов в `try/except`.
4. Не считать любой ответ JSON: сначала понимать контракт API или проверить тип содержимого.
5. Не хранить API-ключи в исходном коде, который будет опубликован.

**Мини-задание 4 (2 минуты).** Измените адрес в примере на `https://httpbin.org/status/200`. Какие строки теперь выполнит программа?

---
## 6. Самостоятельная практика

Выполняйте задания по порядку. Используйте `requests`, а для обработки — уже изученные списки, словари, циклы и файлы. При сетевом запросе ставьте `timeout`, а перед разбором успешного ответа проверяйте статус.

### Задача 1. Исследование HTTP-ответа

Отправьте GET-запрос к `https://httpbin.org/get` и выведите:
- итоговый URL;
- HTTP-статус;
- заголовок `Content-Type`;
- размер тела ответа в байтах.

### Задача 2. Параметры запроса

Передайте серверу `https://httpbin.org/get` параметры `subject=Python`, `lesson=10`, `group=1`. Выведите итоговый URL и значения параметров из JSON-ответа. Определите тип параметра `lesson`, полученного сервером.

### Задача 3. Список задач

Получите задачи пользователя с `userId=3` по адресу `https://jsonplaceholder.typicode.com/todos`. Определите:
- общее число задач;
- количество выполненных и невыполненных задач;
- процент выполненных задач (округлить до 1 знака).

### Задача 4. Статистика по пользователям

Получите **все** задачи с `https://jsonplaceholder.typicode.com/todos`. Составьте словарь вида `{userId: число выполненных задач}` и выведите его по возрастанию `userId`. Найдите пользователя с наибольшим числом выполненных задач (если таких несколько — выведите всех).

### Задача 5. Сохранение результатов

Для задач пользователя `userId=3` создайте локальный файл `todos_user3.json`, содержащий **только невыполненные** задачи. Используйте `json.dump(..., ensure_ascii=False, indent=2)`. Прочитайте файл обратно через `json.load()` и проверьте число записей.

**Дополнительная задача.** Отправьте POST-запрос с новой учебной задачей в JSONPlaceholder, проверьте статус и сравните отправленные и возвращённые поля. Почему запрос нельзя использовать как постоянное хранилище?

In [ ]:
# Задача 1. Напишите решение здесь.

In [ ]:
# Задача 2. Напишите решение здесь.

In [ ]:
# Задача 3. Напишите решение здесь.

In [ ]:
# Задача 4. Напишите решение здесь.

In [ ]:
# Задача 5. Напишите решение здесь.

---
## Дополнительно: знакомство с `socket`

`requests` работает на высоком уровне: она формирует HTTP-запросы и обрабатывает множество деталей. Модуль `socket` предоставляет более низкоуровневый доступ к сетевым соединениям.

### Дополнительные материалы

- [Документация Requests](https://requests.readthedocs.io/en/latest/)
- [HTTP: обзор на MDN](https://developer.mozilla.org/en-US/docs/Web/HTTP/Overview)
- [JSONPlaceholder — учебное REST API](https://jsonplaceholder.typicode.com/)
- [httpbin — сервис для изучения HTTP](https://httpbin.org/)
